# Pixels to Predictions — SmolVLM-500M MCQA

End-to-end notebook for the DL Vision Challenge. We fine-tune **HuggingFaceTB/SmolVLM-500M-Instruct** with a tiny LoRA adapter (well under the 5M trainable-parameter cap) and answer each multiple-choice question by scoring the *letter* tokens (A/B/C/D/E) under the model's vision-language head. This handles the variable choice count (2–5) cleanly and needs no extra classifier weights.

**Pipeline**
1. Load CSVs and resolve image paths.
2. Build prompts: question + hint + lecture + lettered choices, with the image.
3. **Train** (one short epoch with LoRA on the language model's q/k/v/o projections only) — supervised on the correct letter token.
4. **Predict** by scoring each choice's letter token log-probability and taking argmax over the *valid* letters for that question.
5. Write `submission.csv`.

Designed to run on a single Kaggle T4/P100 (or Colab T4) within free-tier limits. If you want a zero-training baseline (still strong), set `DO_TRAIN = False` in the config cell.

## 1. Configuration

Edit the paths if your Kaggle dataset name differs. The defaults assume the competition data is attached at `/kaggle/input/pixels-to-predictions/` with the layout described in the task.

In [ ]:
import os
print("Current dir:", os.getcwd())
print("\nFiles in current dir:")
for f in sorted(os.listdir('.')):
    print(" ", f, "(dir)" if os.path.isdir(f) else "")

# If there's an images folder, check what's inside
if os.path.exists('images'):
    print("\nInside ./images:")
    for f in sorted(os.listdir('images')):
        print(" ", f, "(dir)" if os.path.isdir(os.path.join('images', f)) else "")

Current dir: /content

Files in current dir:
  .config (dir)
  sample_data (dir)


Saving pixels-to-predictions.zip to pixels-to-predictions.zip
Done. /content now contains:
  .config
  images
  pixels-to-predictions.zip
  sample_data
  sample_submission.csv
  test.csv
  train.csv
  val.csv


In [ ]:
import os, sys, json, math, random, gc, time
from pathlib import Path

# --- Paths ----------------------------------------------------------------
# On Kaggle the competition data is typically under /kaggle/input/<competition-name>/
# Adjust DATA_ROOT if your attached dataset has a different folder name.
CANDIDATE_ROOTS = [
    "/kaggle/input/pixels-to-predictions",
    "/kaggle/input/pixels-to-predictions-dl-vision-challenge",
    "/content/pixels-to-predictions",
    "./pixels-to-predictions",
    ".",
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(os.path.join(p, "train.csv"))), None)
assert DATA_ROOT is not None, f"Could not find train.csv under any of {CANDIDATE_ROOTS}. Set DATA_ROOT manually."
print("DATA_ROOT =", DATA_ROOT)

# Image paths in the CSV are like "images/train/train_xxxxx.png".
# In some packagings the actual files live at "<root>/images/<split>/..." (one level)
# and in others at "<root>/images/images/<split>/..." (nested). We auto-detect.
def resolve_image_root(data_root):
    candidates = [
        os.path.join(data_root, "images", "train"),                # CSV path "images/train/x.png" -> data_root + path
        os.path.join(data_root, "images", "images", "train"),      # CSV path resolves under data_root/images/
    ]
    if os.path.exists(candidates[0]):
        return data_root                                            # prefix = data_root
    if os.path.exists(candidates[1]):
        return os.path.join(data_root, "images")                   # prefix = data_root/images
    raise FileNotFoundError("Cannot locate image directories.")
IMAGE_PREFIX = resolve_image_root(DATA_ROOT)
print("IMAGE_PREFIX =", IMAGE_PREFIX)

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model ----------------------------------------------------------------
MODEL_ID   = "HuggingFaceTB/SmolVLM-500M-Instruct"
MAX_IMG_SIZE = 384                  # downsize images for speed/memory
MAX_TEXT_CHARS = 1500                # truncate very long lecture/hint text

# --- Training -------------------------------------------------------------
DO_TRAIN          = True             # set False for a zero-training baseline
NUM_EPOCHS        = 1
BATCH_SIZE        = 1                # SmolVLM-500M with images fits ~1 per step on T4
GRAD_ACCUM_STEPS  = 8                # effective batch size 8
LR                = 2e-4
WEIGHT_DECAY      = 0.0
WARMUP_RATIO      = 0.03
MAX_TRAIN_SAMPLES = None             # set to e.g. 1500 for a quick smoke run
SEED              = 42

# --- LoRA (keeps trainable params well below the 5M cap) -----------------
LORA_R            = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.05
LORA_TARGETS      = ["q_proj", "k_proj", "v_proj", "o_proj"]   # LM attention only

# --- Inference ------------------------------------------------------------
EVAL_BATCH_SIZE   = 1
EVAL_VAL          = True             # evaluate on val.csv after training

random.seed(SEED)
import numpy as np; np.random.seed(SEED)
print("Config loaded.")

DATA_ROOT = .
IMAGE_PREFIX = ./images
Config loaded.


## 2. Install / verify dependencies

Kaggle's default image already has `torch`, `transformers`, `pillow`, `pandas`. We pin minimum versions that include SmolVLM support and add `peft` for LoRA. The `pip install` runs only if a version is missing or too old, so re-running the notebook is fast.

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, import_name=None, min_version=None):
    name = import_name or pkg.split("==")[0].split(">=")[0]
    try:
        m = importlib.import_module(name)
        if min_version:
            from packaging.version import Version
            if Version(getattr(m, "__version__", "0")) < Version(min_version):
                raise ImportError
        return
    except Exception:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("transformers>=4.46.0", "transformers", "4.46.0")
ensure("peft>=0.13.0", "peft", "0.13.0")
ensure("accelerate>=0.34.0", "accelerate", "0.34.0")
ensure("pillow", "PIL")
ensure("pandas")
ensure("numpy")

import torch, transformers, peft
print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

torch: 2.10.0+cu128 | transformers: 4.46.3 | peft: 0.13.2
CUDA available: True | device: Tesla T4


## 3. Load CSVs and clean text

We keep the question, hint, and lecture text but truncate aggressively so the prompt fits comfortably in context.

In [ ]:
import pandas as pd, json
import numpy as np

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_ROOT, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))

print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

def parse_choices(s):
    """choices column is JSON-encoded; fall back to a forgiving parse."""
    if isinstance(s, list): return s
    try:    return json.loads(s)
    except Exception:
        try: return json.loads(s.replace("\\", "\\\\"))
        except Exception:
            return [c.strip() for c in str(s).strip("[]").split(",")]

for df in (train_df, val_df, test_df):
    df["choices_list"] = df["choices"].apply(parse_choices)
    # Defensive: re-derive num_choices in case it disagrees with the parsed list
    df["num_choices"] = df["choices_list"].apply(len)

print("num_choices distribution (train):", train_df.num_choices.value_counts().to_dict())
if "answer" in train_df.columns:
    print("answer distribution (train):", train_df.answer.value_counts().sort_index().to_dict())

def clean(t):
    if t is None or (isinstance(t, float) and math.isnan(t)): return ""
    t = str(t).strip()
    if len(t) > MAX_TEXT_CHARS:
        t = t[:MAX_TEXT_CHARS] + " ..."
    return t

train_df["hint_c"]    = train_df["hint"].apply(clean)
train_df["lecture_c"] = train_df["lecture"].apply(clean)
val_df["hint_c"]      = val_df["hint"].apply(clean)
val_df["lecture_c"]   = val_df["lecture"].apply(clean)
test_df["hint_c"]     = test_df["hint"].apply(clean)
test_df["lecture_c"]  = test_df["lecture"].apply(clean)

# Resolve image path on disk
def resolve_path(rel_path):
    # CSV gives "images/<split>/<file>". Prefix is either DATA_ROOT or DATA_ROOT/images.
    p = os.path.join(IMAGE_PREFIX, rel_path)
    if os.path.exists(p): return p
    # Fallback: maybe IMAGE_PREFIX already includes "images"
    p2 = os.path.join(IMAGE_PREFIX, rel_path.split("/", 1)[-1])
    if os.path.exists(p2): return p2
    return None

for df in (train_df, val_df, test_df):
    df["image_full"] = df["image_path"].apply(resolve_path)
missing = [df["image_full"].isna().sum() for df in (train_df, val_df, test_df)]
print("Missing images train/val/test:", missing)
assert sum(missing) == 0, "Some images could not be located. Check IMAGE_PREFIX."

train: 3109 | val: 1048 | test: 1008
num_choices distribution (train): {3: 1552, 4: 783, 2: 664, 5: 110}
answer distribution (train): {0: 1124, 1: 1028, 2: 737, 3: 204, 4: 16}
Missing images train/val/test: [np.int64(0), np.int64(0), np.int64(0)]


## 4. Load the SmolVLM-500M-Instruct model and processor

We load in fp16 to fit comfortably on a T4. The processor handles image preprocessing and chat-formatted prompts.

In [ ]:
import transformers
print("transformers:", transformers.__version__)
from transformers import AutoModelForVision2Seq
print("import OK")

transformers: 4.46.3
import OK


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

True Tesla T4


In [ ]:
import pandas as pd, json
import numpy as np

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_ROOT, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

def parse_choices(s):
    if isinstance(s, list): return s
    try: return json.loads(s)
    except Exception:
        try: return json.loads(s.replace("\\", "\\\\"))
        except Exception:
            return [c.strip() for c in str(s).strip("[]").split(",")]

for df in (train_df, val_df, test_df):
    df["choices_list"] = df["choices"].apply(parse_choices)
    df["num_choices"] = df["choices_list"].apply(len)

print("num_choices distribution (train):", train_df.num_choices.value_counts().to_dict())
if "answer" in train_df.columns:
    print("answer distribution (train):", train_df.answer.value_counts().sort_index().to_dict())

def clean(t):
    if t is None or (isinstance(t, float) and math.isnan(t)): return ""
    t = str(t).strip()
    if len(t) > MAX_TEXT_CHARS: t = t[:MAX_TEXT_CHARS] + " ..."
    return t

for df in (train_df, val_df, test_df):
    df["hint_c"] = df["hint"].apply(clean)
    df["lecture_c"] = df["lecture"].apply(clean)

def resolve_path(rel_path):
    p = os.path.join(IMAGE_PREFIX, rel_path)
    if os.path.exists(p): return p
    p2 = os.path.join(IMAGE_PREFIX, rel_path.split("/", 1)[-1])
    if os.path.exists(p2): return p2
    return None

for df in (train_df, val_df, test_df):
    df["image_full"] = df["image_path"].apply(resolve_path)
missing = [df["image_full"].isna().sum() for df in (train_df, val_df, test_df)]
print("Missing images train/val/test:", missing)
assert sum(missing) == 0, "Some images could not be located. Check IMAGE_PREFIX."

train: 3109 | val: 1048 | test: 1008
num_choices distribution (train): {3: 1552, 4: 783, 2: 664, 5: 110}
answer distribution (train): {0: 1124, 1: 1028, 2: 737, 3: 204, 4: 16}
Missing images train/val/test: [np.int64(0), np.int64(0), np.int64(0)]


In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch, gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device} | dtype: {dtype}")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total SmolVLM params: {total_params/1e6:.1f}M (frozen baseline)")

tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

LETTERS = ["A", "B", "C", "D", "E"]
def letter_token_id(letter):
    ids = tokenizer.encode(f" {letter}", add_special_tokens=False)
    return ids[-1]
LETTER_IDS = [letter_token_id(l) for l in LETTERS]
print("Letter token ids:", dict(zip(LETTERS, LETTER_IDS)))
assert len(set(LETTER_IDS)) == len(LETTER_IDS), f"Letter token ids collide: {LETTER_IDS}"

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {used:.2f} / {total:.1f} GB used")

Device: cuda | dtype: torch.float16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Total SmolVLM params: 507.5M (frozen baseline)
Letter token ids: {'A': 330, 'B': 389, 'C': 340, 'D': 422, 'E': 414}
GPU memory: 1.04 / 15.6 GB used


## 5. Attach a small LoRA adapter

We freeze the entire base model and train a LoRA adapter on the language-model attention projections only. With `r=8` over 4 projection types in ~30 transformer layers this stays well under the 5M trainable-parameter cap.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Reset any prior LoRA
if hasattr(model, "peft_config"):
    model = model.unload() if hasattr(model, "unload") else model

for p in model.parameters():
    p.requires_grad = False

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", target_modules=LORA_TARGETS, task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,} ({trainable/1e6:.3f}M)")
assert trainable < 5_000_000, f"Trainable {trainable:,} exceeds 5M cap. Drop LORA_R."

Trainable params: 4,784,128 (4.784M)


In [ ]:
processor.image_processor.size = {"longest_edge": 512}
processor.image_processor.max_image_size = {"longest_edge": 512}
print("image size →", processor.image_processor.size)

def load_image(path):
    return Image.open(path).convert("RGB")

image size → {'longest_edge': 512}


## 6. Prompt construction

For each example we build a chat-formatted prompt:

```
<image>
Question: ...
Hint: ...        (omitted if empty)
Lecture: ...     (omitted if empty)
Choices:
A. ...
B. ...
...
Answer:
```

The model is then asked to produce a single letter. During training we supervise only the letter token; during inference we read the next-token logits over `{A, B, C, D, E}` and pick the argmax restricted to the letters that exist for this question.

In [ ]:
def build_user_text(row):
    q = str(row["question"]).strip()
    parts = [f"Question: {q}"]
    h = row.get("hint_c", "")
    if h: parts.append(f"Hint: {h}")
    L = row.get("lecture_c", "")
    if L: parts.append(f"Lecture: {L}")
    parts.append("Choices:")
    for i, c in enumerate(row["choices_list"]):
        parts.append(f"{LETTERS[i]}. {str(c).strip()}")
    parts.append("Answer with the single letter of the best choice.")
    return "\n".join(parts)

def build_chat_messages(row):
    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": build_user_text(row)},
        ],
    }]

## 7. Datasets and collator

We build a PyTorch `Dataset` that emits `(image, prompt_text, target_letter_id)` tuples; the collator runs the processor and adds labels that mask out everything except the assistant's letter token.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class MCQADataset(Dataset):
    def __init__(self, df, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.with_labels = with_labels and ("answer" in df.columns)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["image_full"])
        msgs  = build_chat_messages(row)
        prompt_text = processor.apply_chat_template(msgs, add_generation_prompt=True)
        item = {"image": image, "prompt_text": prompt_text, "row_idx": idx,
                "num_choices": int(row["num_choices"]),
                "choices_list": row["choices_list"]}
        if self.with_labels:
            ans = int(row["answer"])
            item["answer_idx"] = ans
            item["letter_id"]  = LETTER_IDS[ans]
        return item

def collate_train(batch):
    full_texts, images = [], []
    for b in batch:
        full_texts.append(b["prompt_text"] + LETTERS[b["answer_idx"]])
        images.append([b["image"]])
    enc = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    input_ids = enc["input_ids"]
    labels = torch.full_like(input_ids, -100)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    for i in range(input_ids.size(0)):
        nonpad = (input_ids[i] != pad_id).nonzero(as_tuple=True)[0]
        last = nonpad[-1].item()
        labels[i, last] = input_ids[i, last].item()
    enc["labels"] = labels
    return enc

def collate_eval(batch):
    prompt_texts = [b["prompt_text"] for b in batch]
    images       = [[b["image"]] for b in batch]
    enc = processor(text=prompt_texts, images=images, return_tensors="pt", padding=True)
    enc["_num_choices"] = torch.tensor([b["num_choices"] for b in batch], dtype=torch.long)
    enc["_row_idx"]     = torch.tensor([b["row_idx"] for b in batch], dtype=torch.long)
    return enc

train_ds = MCQADataset(train_df, with_labels=True)
val_ds   = MCQADataset(val_df,   with_labels=True)
test_ds  = MCQADataset(test_df,  with_labels=False)

print(f"train_ds={len(train_ds)} val_ds={len(val_ds)} test_ds={len(test_ds)}")

train_ds=3109 val_ds=1048 test_ds=1008


## 8. Training loop

A single short epoch with gradient accumulation. We log loss every ~50 optimizer steps. This is a deliberately minimal trainer (no `Trainer` class) because the chat template + image batching with SmolVLM is cleaner to control directly.

In [ ]:
@torch.no_grad()
def predict_indices_letter(dataset, batch_size=1, desc="predict"):
    """Fast inference: score the next-token logits over A/B/C/D/E."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                       collate_fn=collate_eval, num_workers=2, pin_memory=True)
    preds = [None] * len(dataset)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    letter_ids_t = torch.tensor(LETTER_IDS, device=device)

    t0, n_done = time.time(), 0
    for batch in loader:
        num_choices = batch.pop("_num_choices")
        row_idx     = batch.pop("_row_idx")
        batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
                 for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
            out = model(**batch)
        logits = out.logits
        input_ids = batch["input_ids"]
        for b in range(input_ids.size(0)):
            nonpad = (input_ids[b] != pad_id).nonzero(as_tuple=True)[0]
            last = nonpad[-1].item()
            row_logits = logits[b, last]
            letter_logits = row_logits[letter_ids_t][:int(num_choices[b].item())]
            preds[int(row_idx[b].item())] = int(letter_logits.argmax().item())
        n_done += input_ids.size(0)
        if n_done % 200 == 0:
            print(f"  {desc}: {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")
    return preds

In [ ]:
@torch.no_grad()
def predict_indices_choicetext(dataset, desc="predict-ct"):
    """Score each choice by the model's log-prob of the choice TEXT.
    Slower than letter scoring but more accurate."""
    model.eval()
    preds = [None] * len(dataset)
    t0 = time.time()

    for idx in range(len(dataset)):
        item = dataset[idx]
        image = item["image"]
        prompt_text = item["prompt_text"]
        choices = item["choices_list"]
        nc = item["num_choices"]

        # For each choice, build prompt + " <LETTER>. <choice text>"
        # Score the average log-prob of the choice TEXT tokens.
        choice_logprobs = []
        for k in range(nc):
            full = prompt_text + f"{LETTERS[k]}. {str(choices[k]).strip()}"
            enc = processor(text=[full], images=[[image]],
                            return_tensors="pt", padding=True)
            enc = {kk: vv.to(device) if torch.is_tensor(vv) else vv
                   for kk, vv in enc.items()}

            # Tokenize prompt-only to know where the choice text begins
            prompt_only_enc = processor(text=[prompt_text], images=[[image]],
                                        return_tensors="pt", padding=True)
            prompt_len = (prompt_only_enc["input_ids"] != tokenizer.pad_token_id).sum().item()

            with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
                out = model(**enc)
            logits = out.logits[0]                         # [T, V]
            input_ids = enc["input_ids"][0]
            nonpad_mask = (input_ids != tokenizer.pad_token_id)
            seq_len = nonpad_mask.sum().item()

            # Score positions [prompt_len .. seq_len-1] (these are the choice tokens)
            # logits at position t-1 predict token at position t
            target_positions = list(range(prompt_len, seq_len))
            if not target_positions:
                choice_logprobs.append(float("-inf"))
                continue
            logp = torch.log_softmax(logits.float(), dim=-1)
            scores = []
            for t in target_positions:
                if t == 0: continue
                tok = input_ids[t].item()
                scores.append(logp[t-1, tok].item())
            # Length-normalize so longer choices aren't penalized
            choice_logprobs.append(sum(scores) / max(1, len(scores)))

        preds[idx] = int(max(range(nc), key=lambda k: choice_logprobs[k]))
        if (idx + 1) % 100 == 0:
            print(f"  {desc}: {idx+1}/{len(dataset)} ({time.time()-t0:.0f}s)")
    return preds

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

NUM_EPOCHS = 3
LR = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_RATIO = 0.05

def train_seed(seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model.train()
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                       collate_fn=collate_train, num_workers=2, pin_memory=True,
                       generator=torch.Generator().manual_seed(seed))
    total_steps = math.ceil(len(loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    optim = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=LR, weight_decay=0.01, betas=(0.9, 0.95))
    sched = get_cosine_schedule_with_warmup(
        optim, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    step, t0 = 0, time.time()
    best_val = 0.0
    for epoch in range(NUM_EPOCHS):
        epoch_loss, n_batches = 0.0, 0
        optim.zero_grad()
        model.train()
        for i, batch in enumerate(loader):
            batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
                     for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
                out = model(**batch)
                loss = out.loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            epoch_loss += loss.item() * GRAD_ACCUM_STEPS
            n_batches += 1

            if (i + 1) % GRAD_ACCUM_STEPS == 0 or (i + 1) == len(loader):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optim); scaler.update(); sched.step()
                optim.zero_grad()
                step += 1
                if step % 25 == 0:
                    print(f"seed{seed} ep{epoch+1} step {step:4d} | "
                          f"{i+1}/{len(loader)} | loss {epoch_loss/n_batches:.4f} | "
                          f"lr {sched.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
        print(f"=== seed{seed} ep{epoch+1} avg loss {epoch_loss/n_batches:.4f} | "
              f"total {time.time()-t0:.0f}s ===")

        # Mid-training eval on val subset (200 samples, letter scoring for speed)
        val_subset = torch.utils.data.Subset(val_ds, list(range(min(200, len(val_ds)))))
        preds = predict_indices_letter(val_subset, desc=f"s{seed}-ep{epoch+1}-val")
        truth = [val_df.iloc[i]["answer"] for i in range(len(val_subset))]
        acc = sum(p == t for p, t in zip(preds, truth)) / len(truth)
        print(f"=== seed{seed} ep{epoch+1} val-200 letter-acc: {acc:.4f} ===\n")
        if acc > best_val: best_val = acc
    return best_val

# Train seed 1
print(">>> TRAINING SEED 42")
best = train_seed(42)
print(f"Seed 42 best val-200: {best:.4f}")

# Save the seed-42 adapter
model.save_pretrained(f"{OUTPUT_DIR}/lora_seed42")
print(f"Saved adapter → {OUTPUT_DIR}/lora_seed42")

gc.collect(); torch.cuda.empty_cache()

>>> TRAINING SEED 42


/tmp/ipykernel_14760/466809423.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optim); scaler.update(); sched.step()


seed42 ep1 step   25 | 200/3109 | loss 17.2134 | lr 4.31e-05 | 100s
seed42 ep1 step   50 | 400/3109 | loss 11.9320 | lr 8.62e-05 | 188s
seed42 ep1 step   75 | 600/3109 | loss 8.2211 | lr 9.99e-05 | 274s
seed42 ep1 step  100 | 800/3109 | loss 6.3663 | lr 9.96e-05 | 358s
seed42 ep1 step  125 | 1000/3109 | loss 5.2328 | lr 9.91e-05 | 442s
seed42 ep1 step  150 | 1200/3109 | loss 4.4680 | lr 9.83e-05 | 527s
seed42 ep1 step  175 | 1400/3109 | loss 3.9083 | lr 9.73e-05 | 612s
seed42 ep1 step  200 | 1600/3109 | loss 3.5055 | lr 9.60e-05 | 698s
seed42 ep1 step  225 | 1800/3109 | loss 3.1729 | lr 9.45e-05 | 784s
seed42 ep1 step  250 | 2000/3109 | loss 2.9080 | lr 9.28e-05 | 870s
seed42 ep1 step  275 | 2200/3109 | loss 2.6856 | lr 9.08e-05 | 955s
seed42 ep1 step  300 | 2400/3109 | loss 2.5075 | lr 8.87e-05 | 1040s
seed42 ep1 step  325 | 2600/3109 | loss 2.3575 | lr 8.64e-05 | 1126s
seed42 ep1 step  350 | 2800/3109 | loss 2.2277 | lr 8.38e-05 | 1211s
seed42 ep1 step  375 | 3000/3109 | loss 2.1116 

In [ ]:
print(">>> Full validation with choice-text scoring (slower, more accurate)")
val_preds_ct = predict_indices_choicetext(val_ds, desc="val-ct")
y_true = val_df["answer"].astype(int).tolist()
acc_ct = float(np.mean([p == t for p, t in zip(val_preds_ct, y_true)]))
print(f"\nChoice-text val accuracy: {acc_ct:.4f}")

# Also do letter scoring for comparison
val_preds_letter = predict_indices_letter(val_ds, desc="val-letter")
acc_letter = float(np.mean([p == t for p, t in zip(val_preds_letter, y_true)]))
print(f"Letter val accuracy:      {acc_letter:.4f}")

# Pick whichever is better, or blend
USE_BLEND = True   # set False to use whichever single method scored higher

>>> Full validation with choice-text scoring (slower, more accurate)
  val-ct: 100/1048 (77s)
  val-ct: 200/1048 (136s)
  val-ct: 300/1048 (194s)
  val-ct: 400/1048 (262s)
  val-ct: 500/1048 (320s)
  val-ct: 600/1048 (379s)
  val-ct: 700/1048 (440s)
  val-ct: 800/1048 (497s)
  val-ct: 900/1048 (574s)
  val-ct: 1000/1048 (638s)

Choice-text val accuracy: 0.6746
  val-letter: 200/1048 (33s)
  val-letter: 400/1048 (66s)
  val-letter: 600/1048 (97s)
  val-letter: 800/1048 (131s)
  val-letter: 1000/1048 (163s)
Letter val accuracy:      0.7538


In [ ]:
# Reload base model fresh and re-attach LoRA with new seed
del model
gc.collect(); torch.cuda.empty_cache()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True,
    attn_implementation="eager",
).to(device)
for p in model.parameters(): p.requires_grad = False

lora_config_2 = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", target_modules=LORA_TARGETS, task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config_2)

print(">>> TRAINING SEED 7")
best2 = train_seed(7)
print(f"Seed 7 best val-200: {best2:.4f}")
model.save_pretrained(f"{OUTPUT_DIR}/lora_seed7")
gc.collect(); torch.cuda.empty_cache()

>>> TRAINING SEED 7


/tmp/ipykernel_14760/466809423.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optim); scaler.update(); sched.step()


seed7 ep1 step   25 | 200/3109 | loss 16.9347 | lr 4.31e-05 | 86s
seed7 ep1 step   50 | 400/3109 | loss 12.0327 | lr 8.62e-05 | 171s
seed7 ep1 step   75 | 600/3109 | loss 8.3057 | lr 9.99e-05 | 257s
seed7 ep1 step  100 | 800/3109 | loss 6.4009 | lr 9.96e-05 | 342s
seed7 ep1 step  125 | 1000/3109 | loss 5.2565 | lr 9.91e-05 | 424s
seed7 ep1 step  150 | 1200/3109 | loss 4.4850 | lr 9.83e-05 | 507s
seed7 ep1 step  175 | 1400/3109 | loss 3.9576 | lr 9.73e-05 | 591s
seed7 ep1 step  200 | 1600/3109 | loss 3.5515 | lr 9.60e-05 | 674s
seed7 ep1 step  225 | 1800/3109 | loss 3.2180 | lr 9.45e-05 | 758s
seed7 ep1 step  250 | 2000/3109 | loss 2.9556 | lr 9.28e-05 | 842s
seed7 ep1 step  275 | 2200/3109 | loss 2.7372 | lr 9.08e-05 | 927s
seed7 ep1 step  300 | 2400/3109 | loss 2.5425 | lr 8.87e-05 | 1012s
seed7 ep1 step  325 | 2600/3109 | loss 2.3907 | lr 8.64e-05 | 1098s
seed7 ep1 step  350 | 2800/3109 | loss 2.2567 | lr 8.38e-05 | 1182s
seed7 ep1 step  375 | 3000/3109 | loss 2.1442 | lr 8.12e-05 | 

In [ ]:
# Use the seed-42 model that's currently loaded.
# Choice-text scoring on test:
test_preds = predict_indices_choicetext(test_ds, desc="test-ct")
print(f"Got {len(test_preds)} test predictions.")

  test-ct: 100/1008 (67s)
  test-ct: 200/1008 (125s)
  test-ct: 300/1008 (179s)
  test-ct: 400/1008 (240s)
  test-ct: 500/1008 (299s)
  test-ct: 600/1008 (352s)
  test-ct: 700/1008 (407s)
  test-ct: 800/1008 (466s)
  test-ct: 900/1008 (531s)
  test-ct: 1000/1008 (605s)
Got 1008 test predictions.


In [ ]:
@torch.no_grad()
def score_test_choicetext_logprobs(dataset):
    """Return per-example, per-choice log-probs (variable shape). For ensembling."""
    model.eval()
    all_scores = []
    t0 = time.time()
    for idx in range(len(dataset)):
        item = dataset[idx]
        image, prompt_text = item["image"], item["prompt_text"]
        choices, nc = item["choices_list"], item["num_choices"]
        choice_logprobs = []
        for k in range(nc):
            full = prompt_text + f"{LETTERS[k]}. {str(choices[k]).strip()}"
            enc = processor(text=[full], images=[[image]], return_tensors="pt", padding=True)
            enc = {kk: vv.to(device) if torch.is_tensor(vv) else vv for kk, vv in enc.items()}
            prompt_only_enc = processor(text=[prompt_text], images=[[image]], return_tensors="pt", padding=True)
            prompt_len = (prompt_only_enc["input_ids"] != tokenizer.pad_token_id).sum().item()
            with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
                out = model(**enc)
            logits = out.logits[0]
            input_ids = enc["input_ids"][0]
            seq_len = (input_ids != tokenizer.pad_token_id).sum().item()
            logp = torch.log_softmax(logits.float(), dim=-1)
            scores = []
            for t in range(prompt_len, seq_len):
                if t == 0: continue
                scores.append(logp[t-1, input_ids[t].item()].item())
            choice_logprobs.append(sum(scores) / max(1, len(scores)))
        all_scores.append(choice_logprobs)
        if (idx+1) % 200 == 0: print(f"  scoring: {idx+1}/{len(dataset)} ({time.time()-t0:.0f}s)")
    return all_scores

# Score with currently loaded model (seed 7 from §12)
print(">>> Scoring test with seed 7")
scores_s7 = score_test_choicetext_logprobs(test_ds)

# Reload seed-42 adapter and score
from peft import PeftModel
del model; gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True,
    attn_implementation="eager",
).to(device)
model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/lora_seed42").to(device)
print(">>> Scoring test with seed 42")
scores_s42 = score_test_choicetext_logprobs(test_ds)

# Average per-choice log-probs and pick argmax
test_preds = []
for s7, s42 in zip(scores_s7, scores_s42):
    avg = [(a + b) / 2 for a, b in zip(s7, s42)]
    test_preds.append(int(max(range(len(avg)), key=lambda k: avg[k])))
print(f"Got {len(test_preds)} ensembled test predictions.")

>>> Scoring test with seed 7
  scoring: 200/1008 (125s)
  scoring: 400/1008 (238s)
  scoring: 600/1008 (352s)
  scoring: 800/1008 (464s)
  scoring: 1000/1008 (599s)
>>> Scoring test with seed 42
  scoring: 200/1008 (125s)
  scoring: 400/1008 (239s)
  scoring: 600/1008 (354s)
  scoring: 800/1008 (471s)
  scoring: 1000/1008 (613s)
Got 1008 ensembled test predictions.


In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"].tolist(),
    "answer": [int(p) for p in test_preds],
})

# Defensive: clamp out-of-range
nc = test_df["num_choices"].astype(int).tolist()
fixed = 0
for i, (a, k) in enumerate(zip(submission["answer"], nc)):
    if not (0 <= a < k):
        submission.at[i, "answer"] = 0; fixed += 1
if fixed: print(f"Clamped {fixed} predictions.")

# Match sample order
sample_sub_path = os.path.join(DATA_ROOT, "sample_submission.csv")
if os.path.exists(sample_sub_path):
    samp = pd.read_csv(sample_sub_path)
    submission = submission.set_index("id").loc[samp["id"]].reset_index()

out_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(submission)} rows)")
print(submission.head())
print("Answer dist:", submission["answer"].value_counts().sort_index().to_dict())

# Download to local
from google.colab import files
files.download(out_path)

Wrote ./submission.csv (1008 rows)
           id  answer
0  test_01750       2
1  test_00128       1
2  test_02891       4
3  test_02425       1
4  test_00930       4
Answer dist: {0: 352, 1: 342, 2: 219, 3: 74, 4: 21}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Inference: score the letter tokens

We feed the prompt (no answer appended) and read the logits at the final non-pad position. Among the letters valid for that example (`A` … letter for `num_choices-1`), we pick the highest-probability letter.

In [ ]:
@torch.no_grad()
def predict_indices(dataset, batch_size=EVAL_BATCH_SIZE, desc="predict"):
    model.eval()
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_eval, num_workers=2, pin_memory=True,
    )
    preds = [None] * len(dataset)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    letter_ids_t = torch.tensor(LETTER_IDS, device=device)

    t0 = time.time()
    n_done = 0
    for batch in loader:
        num_choices = batch.pop("_num_choices")
        row_idx     = batch.pop("_row_idx")
        batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
            out = model(**batch)
        logits = out.logits  # [B, T, V]
        input_ids = batch["input_ids"]
        for b in range(input_ids.size(0)):
            nonpad = (input_ids[b] != pad_id).nonzero(as_tuple=True)[0]
            last = nonpad[-1].item()
            # Next-token logits live AT position `last` (predicting token last+1).
            row_logits = logits[b, last]
            letter_logits = row_logits[letter_ids_t]            # length 5
            nc = int(num_choices[b].item())
            letter_logits = letter_logits[:nc]                  # restrict to valid letters
            pred = int(letter_logits.argmax().item())
            preds[int(row_idx[b].item())] = pred
        n_done += input_ids.size(0)
        if n_done % 100 == 0:
            print(f"  {desc}: {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")
    return preds

## 10. Validation accuracy

In [ ]:
if EVAL_VAL:
    val_preds = predict_indices(val_ds, desc="val")
    y_true = val_df["answer"].astype(int).tolist()
    acc = float(np.mean([p == t for p, t in zip(val_preds, y_true)]))
    print(f"\nValidation accuracy: {acc:.4f}  ({sum(p==t for p,t in zip(val_preds,y_true))}/{len(y_true)})")
    # Per-num-choices breakdown
    val_df_eval = val_df.copy()
    val_df_eval["pred"] = val_preds
    print("\nAccuracy by num_choices:")
    print(val_df_eval.groupby("num_choices").apply(lambda d: (d["pred"]==d["answer"]).mean()).round(3))
else:
    print("Skipping validation.")

  val: 100/1048 (14s)
  val: 200/1048 (28s)
  val: 300/1048 (42s)
  val: 400/1048 (56s)
  val: 500/1048 (70s)
  val: 600/1048 (84s)
  val: 700/1048 (98s)
  val: 800/1048 (112s)
  val: 900/1048 (127s)
  val: 1000/1048 (141s)

Validation accuracy: 0.6918  (725/1048)

Accuracy by num_choices:
num_choices
2    0.816
3    0.650
4    0.754
5    0.136
dtype: float64


/tmp/ipykernel_14760/2764864110.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(val_df_eval.groupby("num_choices").apply(lambda d: (d["pred"]==d["answer"]).mean()).round(3))


## 11. Test predictions and submission file

In [ ]:
test_preds = predict_indices(test_ds, desc="test")

submission = pd.DataFrame({
    "id": test_df["id"].tolist(),
    "answer": [int(p) for p in test_preds],
})

# Defensive: ensure every predicted index is valid for its question's choice set
nc = test_df["num_choices"].astype(int).tolist()
fixed = 0
for i, (a, k) in enumerate(zip(submission["answer"], nc)):
    if not (0 <= a < k):
        submission.at[i, "answer"] = 0
        fixed += 1
if fixed:
    print(f"Clamped {fixed} out-of-range predictions to 0.")

# Sanity vs sample_submission
sample_sub_path = os.path.join(DATA_ROOT, "sample_submission.csv")
if os.path.exists(sample_sub_path):
    samp = pd.read_csv(sample_sub_path)
    assert set(samp["id"]) == set(submission["id"]), "Submission ids do not match sample_submission ids."
    submission = submission.set_index("id").loc[samp["id"]].reset_index()  # match order
    assert list(submission.columns) == ["id", "answer"]

out_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Wrote {out_path}  ({len(submission)} rows)")
print(submission.head())
print("\nAnswer distribution:", submission["answer"].value_counts().sort_index().to_dict())

  test: 100/1008 (15s)
  test: 200/1008 (29s)
  test: 300/1008 (42s)
  test: 400/1008 (56s)
  test: 500/1008 (70s)
  test: 600/1008 (84s)
  test: 700/1008 (98s)
  test: 800/1008 (112s)
  test: 900/1008 (126s)
  test: 1000/1008 (140s)
Wrote ./submission.csv  (1008 rows)
           id  answer
0  test_01750       1
1  test_00128       1
2  test_02891       3
3  test_02425       1
4  test_00930       3

Answer distribution: {0: 300, 1: 369, 2: 233, 3: 93, 4: 13}


## 12. (Optional) Save the LoRA adapter

Useful if you want to re-run inference later without retraining. Skip if you don't need it.

In [ ]:
if DO_TRAIN:
    adapter_dir = os.path.join(OUTPUT_DIR, "lora_adapter")
    model.save_pretrained(adapter_dir)
    print(f"Saved LoRA adapter to {adapter_dir}")
else:
    print("No adapter to save (DO_TRAIN=False).")

## Notes & tuning tips

- **Trainable-parameter budget.** With `LORA_R=8` on `q,k,v,o` of the language model attention only, you'll see well under 5M trainable params. The cell in §5 hard-asserts this so you can't accidentally exceed it.
- **Rules-compliant offline run.** When the grader runs this notebook offline, the `pip install` cells become no-ops if the right versions are pre-installed in the Kaggle image. The model itself is downloaded from HuggingFace at training time and cached; for a strictly-offline rerun, attach the model as a Kaggle dataset and point `MODEL_ID` at that local path.
- **If you're memory-constrained.** Drop `MAX_IMG_SIZE` to 320, keep `BATCH_SIZE=1`, raise `GRAD_ACCUM_STEPS` to 16. SmolVLM-500M with `bfloat16/float16` and image size 384 fits on a T4 with this config.
- **If you have time to spare.** Increase `NUM_EPOCHS` to 2–3; the LoRA adapter is small enough to keep learning from this dataset without overfitting catastrophically. Watch the validation accuracy in §10.
- **Zero-training baseline.** Set `DO_TRAIN = False` in §1 to skip fine-tuning entirely. SmolVLM-Instruct already follows MCQA prompts surprisingly well, so this gives you a useful sanity check before committing GPU time.